# Notebook 02: Continuous HMM Fitting and Model Internals

This notebook trains a continuous Gaussian HMM via the Baum-Welch (EM) algorithm on SPY excess growth rates,
and visualizes the fitted model internals.

### Key Difference from the Discrete Paper
The published paper (arXiv:2603.10202) used Laplace quantile binning + frequency counting.
Here we learn emission parameters (means, variances) and transition probabilities directly from
continuous observations via Baum-Welch, eliminating quantization error.

### Outputs
- **Convergence plot**: EM log-likelihood history
- **Figure S1 equivalent**: Emission distributions, transition matrix heatmap, residence times

## Setup

In [ ]:
include("../Include.jl");

In [ ]:
# --- CONSTANTS ---
risk_free_rate = 0.0;
Δt = 1/252;
ticker = "SPY";
number_of_states = 13;
max_iter = 60;

## Load Data and Compute Returns

In [ ]:
# --- LOAD DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_excess_return_matrix = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
Rᵢ = findfirst(x -> x == ticker, list_of_all_tickers) |> i -> all_firms_excess_return_matrix[:, i];
in_sample_data = Rᵢ[1:(maximum_number_trading_days-1)];

println("Training data: $(length(in_sample_data)) observations for $ticker")

## Train Continuous HMM via Baum-Welch

In [ ]:
# --- TRAIN MODEL ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = in_sample_data,
    number_of_states = number_of_states,
    max_iter = max_iter
));

println("Training complete: $(length(model.log_likelihood_history)) iterations")

In [ ]:
# --- CONVERGENCE PLOT ---
p_conv = plot(model.log_likelihood_history,
    title="Baum-Welch Convergence ($(ticker), K=$(number_of_states))",
    xlabel="Iteration", ylabel="Log-Likelihood",
    legend=false, lw=2, color=:navy, marker=:circle, ms=3);
display(p_conv)

savefig(p_conv, joinpath(_PATH_TO_FIGURES, "Fig-BW-Convergence-$(ticker).svg"))

## Inspect Learned Emission Distributions

Each hidden state has a Gaussian emission distribution N(μₖ, σₖ).
States are ordered by mean (state 1 = most negative = crash regime).

In [ ]:
# --- EMISSION PARAMETERS TABLE ---
K = length(model.states);

println("State | Mean (μ)     | Std Dev (σ)  | Interpretation")
println("------+--------------+--------------+---------------")
for s in 1:K
    d = model.emission[s];
    μ_s = round(mean(d), digits=2);
    σ_s = round(std(d), digits=2);
    label = if s <= 3 "CRASH" elseif s >= K-2 "BOOM" else "" end;
    println("  $(lpad(s,2))  | $(lpad(μ_s,12)) | $(lpad(σ_s,12)) | $label")
end

In [ ]:
# --- EMISSION PDFs OVERLAY ---
x_grid = range(-800, 800, length=2000);

# Color gradient: crash (red/purple) -> neutral (gray) -> boom (green/blue)
colors = cgrad(:RdYlBu, K, categorical=true);

p_emit = plot(title="Learned Emission Distributions (K=$K)", titlefontsize=10,
    xlabel="Excess Growth Rate", ylabel="Density", legend=:topright);

# Observed histogram
histogram!(p_emit, in_sample_data, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");

# Overlay each state's emission PDF
for s in 1:K
    d = model.emission[s];
    plot!(p_emit, x_grid, pdf.(d, x_grid), lw=1.5, color=colors[s], label="State $s", alpha=0.8);
end

xlims!(p_emit, -800, 800);
display(p_emit)
savefig(p_emit, joinpath(_PATH_TO_FIGURES, "Fig-Emission-PDFs-$(ticker).svg"))

## Transition Matrix Heatmap

Visualize the learned K x K transition matrix. A dominant near-diagonal band
reflects short-range regime persistence.

In [ ]:
# --- TRANSITION MATRIX ---
T_matrix = zeros(K, K);
for i in 1:K
    T_matrix[i, :] = probs(model.transition[i]);
end

# Heatmap in log10 scale
T_log = log10.(T_matrix .+ 1e-10);  # avoid log(0)

p_trans = heatmap(T_log, title="Transition Matrix (log₁₀ scale)", titlefontsize=10,
    xlabel="To State", ylabel="From State", color=:viridis,
    yflip=true, aspect_ratio=:equal, size=(600,500));
display(p_trans)
savefig(p_trans, joinpath(_PATH_TO_FIGURES, "Fig-Transition-Matrix-$(ticker).svg"))

## Natural Residence Times

The expected residence time in state k is 1/(1 - T_kk). Under pure Markov dynamics,
even tail states exit within 1-2 steps, motivating the jump mechanism.

In [ ]:
# --- RESIDENCE TIMES ---
residence_times = [1.0 / (1.0 - T_matrix[k, k]) for k in 1:K];

p_res = bar(1:K, residence_times, title="Expected Natural Residence Time per State", titlefontsize=10,
    xlabel="State", ylabel="Steps", legend=false, color=:steelblue, alpha=0.7);

# Highlight tail states
bar!(p_res, 1:3, residence_times[1:3], color=:red, alpha=0.5, label="Crash states");
bar!(p_res, (K-2):K, residence_times[(K-2):K], color=:teal, alpha=0.5, label="Boom states");

display(p_res)
savefig(p_res, joinpath(_PATH_TO_FIGURES, "Fig-Residence-Times-$(ticker).svg"))

## Stationary Distribution

Compute π = π T by matrix powering (T^1000).

In [ ]:
# --- STATIONARY DISTRIBUTION ---
π_stationary = (T_matrix^1000)[1, :];

p_pi = bar(1:K, π_stationary, title="Stationary Distribution π", titlefontsize=10,
    xlabel="State", ylabel="Probability", legend=false, color=:steelblue, alpha=0.7);
display(p_pi)

println("Sum of π: $(round(sum(π_stationary), digits=6))")

## Save Model Artifacts

Save the trained model for use in subsequent notebooks.

In [ ]:
# --- SAVE ---
path_to_save = joinpath(_PATH_TO_DATA, "CHMM-$(ticker)-K$(number_of_states)-base.jld2");
save(path_to_save, Dict(
    "model" => model,
    "T_matrix" => T_matrix,
    "pi_stationary" => π_stationary,
    "ticker" => ticker,
    "in_sample_data" => in_sample_data,
    "number_of_states" => number_of_states
));
println("Model saved to: $path_to_save")

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute financial advice.